# Ensemble Methods 🎯

Why use one model when you can combine many? Ensemble methods are some of the most powerful and practical algorithms in machine learning.

## What You'll Learn
- Bagging (Bootstrap Aggregating)
- Random Forests
- Gradient Boosting (XGBoost)
- Voting classifiers
- When to use each method
- Feature importance across ensembles

## The Wisdom of Crowds 👥

### Core Idea

Imagine asking 1 person vs 100 people to guess the number of jellybeans in a jar:
- 1 person: Often very wrong
- 100 people (averaging): Usually closer to correct!

Same principle applies to machine learning:
- 1 model: Can overfit, has biases
- Many models (combined): More robust, better generalization

### Key Ensemble Strategies

1. **Bagging:** Train same algorithm on different data subsets → average predictions
2. **Boosting:** Train models sequentially, each corrects previous mistakes
3. **Stacking:** Train models, use their predictions as input to another model
4. **Voting:** Different algorithms vote on prediction

## Random Forests 🌲🌲🌲

### What It Is

A collection of decision trees, where:
- Each tree trains on random data sample (with replacement)
- Each split considers random features
- Final prediction: Average (regression) or majority vote (classification)

### Why It Works

```
Single Deep Tree:        Random Forest:
  Overfits easily         Many shallow trees
  High variance           Low variance (averaging)
  Unstable                Stable
```

The randomness:
- Reduces overfitting (trees are decorrelated)
- Handles high-dimensional data well
- Works with many features automatically

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns

print("📚 Libraries loaded and ready!")

In [ ]:
# Create dataset
X, y = make_classification(n_samples=300, n_features=10, n_informative=5, 
                           n_redundant=2, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test set: {X_test.shape[0]} samples")

In [ ]:
# Compare: Single Deep Tree vs Random Forest

# Single deep tree (prone to overfitting)
deep_tree = DecisionTreeClassifier(max_depth=20, random_state=42)
deep_tree.fit(X_train, y_train)
deep_tree_train = deep_tree.score(X_train, y_train)
deep_tree_test = deep_tree.score(X_test, y_test)

# Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)
rf_train = rf.score(X_train, y_train)
rf_test = rf.score(X_test, y_test)

print("\n" + "="*60)
print("SINGLE DEEP TREE vs RANDOM FOREST")
print("="*60)

print(f"\n🌳 Single Deep Tree (max_depth=20):")
print(f"   Training accuracy: {deep_tree_train:.1%}")
print(f"   Test accuracy:     {deep_tree_test:.1%}")
print(f"   Overfitting gap:   {deep_tree_train - deep_tree_test:.1%}")

print(f"\n🌲🌲🌲 Random Forest (100 trees, max_depth=10):")
print(f"   Training accuracy: {rf_train:.1%}")
print(f"   Test accuracy:     {rf_test:.1%}")
print(f"   Overfitting gap:   {rf_train - rf_test:.1%}")

print(f"\n✅ Random Forest test accuracy is BETTER!")
print(f"✅ Overfitting is much LOWER!")

In [ ]:
# Effect of number of trees
n_trees_range = [1, 5, 10, 20, 50, 100, 200]
test_scores = []
train_scores = []

for n_trees in n_trees_range:
    rf = RandomForestClassifier(n_estimators=n_trees, max_depth=10, random_state=42)
    rf.fit(X_train, y_train)
    train_scores.append(rf.score(X_train, y_train))
    test_scores.append(rf.score(X_test, y_test))

plt.figure(figsize=(10, 6))
plt.plot(n_trees_range, train_scores, marker='o', linewidth=2, label='Training Accuracy', color='green')
plt.plot(n_trees_range, test_scores, marker='s', linewidth=2, label='Test Accuracy', color='red')
plt.xlabel('Number of Trees', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Random Forest: Effect of Number of Trees', fontsize=14)
plt.xticks(n_trees_range)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("📊 Observations:")
print("- More trees = better performance (up to a point)")
print("- Diminishing returns after ~50-100 trees")
print("- More trees = longer training time (but still fast!)")

In [ ]:
# Feature importance from Random Forest
rf_final = RandomForestClassifier(n_estimators=100, random_state=42)
rf_final.fit(X_train, y_train)

importances = rf_final.feature_importances_
feature_names = [f'Feature {i}' for i in range(X_train.shape[1])]

# Sort by importance
indices = np.argsort(importances)[::-1]

print("\nFeature Importance (Random Forest):")
print("-" * 40)
for i in range(10):
    print(f"{i+1}. {feature_names[indices[i]]}: {importances[indices[i]]:.4f}")

# Visualize
plt.figure(figsize=(10, 6))
plt.barh(range(10), importances[indices[:10]], color='skyblue', edgecolor='black')
plt.yticks(range(10), [feature_names[i] for i in indices[:10]])
plt.xlabel('Importance', fontsize=12)
plt.title('Top 10 Most Important Features', fontsize=14)
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\n💡 Use these to understand what the model focuses on!")

## Gradient Boosting & XGBoost ⚡

### How It's Different from Random Forest

**Random Forest (Bagging):**
- Trees trained independently in parallel
- Each tree is weak (shallow)
- Reduce variance by averaging

**Gradient Boosting:**
- Trees trained sequentially
- Each new tree fixes mistakes of previous
- Each tree is very weak (depth=1)
- Reduce bias by iterative improvement

### Visual Comparison

```
Random Forest:              Gradient Boosting:
Tree 1 ──┐                 Tree 1 (weak)
Tree 2 ──┼→ Average        ↓
Tree 3 ──┘                 Tree 2 (fixes Tree 1 errors)
                           ↓
                           Tree 3 (fixes Tree 2 errors)
                           ↓
                           Final prediction = Sum
```

### Why XGBoost?
- Optimized Gradient Boosting implementation
- Faster training
- Regularization built-in
- Handles missing values
- Parallel processing

In [ ]:
# Install XGBoost if needed
# !pip install xgboost

from xgboost import XGBClassifier

# Train XGBoost
xgb = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, verbosity=0)
xgb.fit(X_train, y_train)

xgb_train = xgb.score(X_train, y_train)
xgb_test = xgb.score(X_test, y_test)

print("\n" + "="*60)
print("GRADIENT BOOSTING (XGBoost)")
print("="*60)

print(f"\nTraining accuracy: {xgb_train:.1%}")
print(f"Test accuracy:     {xgb_test:.1%}")
print(f"Overfitting gap:   {xgb_train - xgb_test:.1%}")

In [ ]:
# Compare all three methods
methods = ['Single Tree\n(Deep)', 'Random Forest\n(100 trees)', 'XGBoost\n(100 rounds)']
test_accs = [deep_tree_test, rf_test, xgb_test]
overfit_gaps = [deep_tree_train - deep_tree_test, rf_train - rf_test, xgb_train - xgb_test]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Test Accuracy
colors = ['coral', 'skyblue', 'lightgreen']
axes[0].bar(methods, test_accs, color=colors, edgecolor='black', linewidth=2)
axes[0].set_ylabel('Test Accuracy', fontsize=12)
axes[0].set_title('Model Comparison: Test Accuracy', fontsize=13)
axes[0].set_ylim([0.7, 1.0])
for i, v in enumerate(test_accs):
    axes[0].text(i, v + 0.01, f'{v:.1%}', ha='center', fontsize=11, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Overfitting Gap
axes[1].bar(methods, overfit_gaps, color=colors, edgecolor='black', linewidth=2)
axes[1].set_ylabel('Overfitting Gap (Train - Test)', fontsize=12)
axes[1].set_title('Model Comparison: Overfitting', fontsize=13)
axes[1].set_ylim([0, 0.3])
for i, v in enumerate(overfit_gaps):
    axes[1].text(i, v + 0.01, f'{v:.1%}', ha='center', fontsize=11, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Voting Classifiers 🗳️

Combine different algorithm types (not just one algorithm):
- Logistic Regression
- Decision Tree
- Random Forest
→ All vote on prediction

### Voting Strategies

- **Hard Voting:** Majority vote (e.g., 2 out of 3 predict class 1)
- **Soft Voting:** Average probability predictions (more sophisticated)

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

# Create different classifiers
lr = LogisticRegression(random_state=42, max_iter=1000)
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
rf = RandomForestClassifier(n_estimators=50, random_state=42)
svm = SVC(kernel='rbf', probability=True, random_state=42)

# Hard Voting
hard_voter = VotingClassifier(
    estimators=[('lr', lr), ('dt', dt), ('rf', rf), ('svm', svm)],
    voting='hard'
)

# Soft Voting
soft_voter = VotingClassifier(
    estimators=[('lr', lr), ('dt', dt), ('rf', rf), ('svm', svm)],
    voting='soft'
)

# Train
hard_voter.fit(X_train, y_train)
soft_voter.fit(X_train, y_train)

# Evaluate
print("\n" + "="*60)
print("VOTING CLASSIFIERS")
print("="*60)

print(f"\nIndividual Models:")
print(f"  Logistic Regression: {lr.score(X_test, y_test):.1%}")
print(f"  Decision Tree:       {dt.score(X_test, y_test):.1%}")
print(f"  Random Forest:       {rf.score(X_test, y_test):.1%}")
print(f"  SVM:                 {svm.score(X_test, y_test):.1%}")

print(f"\nEnsembles:")
print(f"  Hard Voting:         {hard_voter.score(X_test, y_test):.1%}")
print(f"  Soft Voting:         {soft_voter.score(X_test, y_test):.1%}")

print(f"\n✅ Ensemble beats all individual models!")

## Real-World Example: Iris Classification 🌸

In [ ]:
# Load iris dataset
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42
)

# Train Random Forest
rf_iris = RandomForestClassifier(n_estimators=100, random_state=42)
rf_iris.fit(X_train_iris, y_train_iris)

y_pred_iris = rf_iris.predict(X_test_iris)
accuracy_iris = accuracy_score(y_test_iris, y_pred_iris)

print("\nIris Dataset - Random Forest Classification")
print("="*50)
print(f"\nAccuracy: {accuracy_iris:.1%}")
print(f"\nClassification Report:")
print(classification_report(y_test_iris, y_pred_iris, 
                          target_names=iris.target_names))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test_iris, y_pred_iris)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=iris.target_names,
            yticklabels=iris.target_names,
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.title('Confusion Matrix: Iris Classification', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance for iris
importances_iris = rf_iris.feature_importances_
feature_names_iris = iris.feature_names

# Sort
indices_iris = np.argsort(importances_iris)[::-1]

plt.figure(figsize=(10, 6))
plt.barh(range(len(importances_iris)), 
         importances_iris[indices_iris],
         color='lightcoral', edgecolor='black', linewidth=2)
plt.yticks(range(len(importances_iris)), 
           [feature_names_iris[i] for i in indices_iris])
plt.xlabel('Importance', fontsize=12)
plt.title('Feature Importance: Iris Dataset', fontsize=14)
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\nFeature Importance:")
for i in range(len(importances_iris)):
    idx = indices_iris[i]
    print(f"  {feature_names_iris[idx]:25s}: {importances_iris[idx]:.4f}")

## Hyperparameter Tuning 🔧

### Random Forest Parameters

- **n_estimators:** Number of trees (more = better, but diminishing returns)
- **max_depth:** Max tree depth (smaller = simpler, less overfitting)
- **min_samples_split:** Min samples to split node (larger = simpler)
- **min_samples_leaf:** Min samples in leaf (larger = simpler)
- **max_features:** Features to consider at split ('sqrt', 'log2', or number)

### XGBoost Parameters

- **n_estimators:** Number of boosting rounds
- **max_depth:** Max tree depth (usually 3-8)
- **learning_rate:** Step size (smaller = more iterations needed, less overfitting)
- **subsample:** Fraction of training samples per round (0-1)
- **lambda:** L2 regularization (larger = more regularization)

In [ ]:
from sklearn.model_selection import GridSearchCV

# Hyperparameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5, 10]
}

# Grid search
print("\nRunning GridSearchCV...")
print("Testing 3 × 3 × 3 = 27 parameter combinations")
print("-" * 50)

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,  # 5-fold cross-validation
    n_jobs=-1  # Use all CPU cores
)

grid_search.fit(X_train, y_train)

print(f"\n✅ Best parameters found:")
print(f"   {grid_search.best_params_}")
print(f"\nBest cross-validation score: {grid_search.best_score_:.1%}")

# Test with best model
best_model = grid_search.best_estimator_
test_acc_best = best_model.score(X_test, y_test)
print(f"Test accuracy: {test_acc_best:.1%}")

## Choosing the Right Ensemble Method 🎯

In [ ]:
print("\n" + "="*70)
print("ENSEMBLE METHODS COMPARISON")
print("="*70)

comparison = """
┌─────────────────┬──────────────┬─────────────────┬──────────────────┐
│ Method          │ Speed        │ Interpretable   │ Best For         │
├─────────────────┼──────────────┼─────────────────┼──────────────────┤
│ Random Forest   │ ⚡⚡ Fast    │ Medium          │ Quick baseline   │
│                 │              │ (importance)    │ Good accuracy    │
│                 │              │                 │ Parallelizable   │
├─────────────────┼──────────────┼─────────────────┼──────────────────┤
│ XGBoost         │ ⚡ Medium    │ Low             │ Kaggle contests  │
│                 │              │                 │ Production use   │
│                 │              │                 │ Top performance  │
├─────────────────┼──────────────┼─────────────────┼──────────────────┤
│ Voting          │ Slow         │ High            │ Interpretability │
│ Classifier      │              │ (see votes)     │ Production ready │
│                 │              │                 │ Mixed algorithms │
├─────────────────┼──────────────┼─────────────────┼──────────────────┤
│ Gradient        │ ⚡ Medium    │ Low             │ Maximum accuracy │
│ Boosting (LGB)  │              │                 │ (needs tuning)   │
└─────────────────┴──────────────┴─────────────────┴──────────────────┘

QUICK RECOMMENDATION:
  
  1️⃣  START with Random Forest
      - Easy to use
      - Works well with minimal tuning
      - Fast to train
      
  2️⃣  IF you need better accuracy → XGBoost
      - Can squeeze out 1-2% more accuracy
      - Requires more hyperparameter tuning
      - Industry standard for competitions
      
  3️⃣  IF you need interpretability → Voting
      - Combine diverse algorithms
      - Can explain individual model votes
      - Production-friendly
      
  4️⃣  For maximum accuracy → Stacking
      - Combine multiple ensemble methods
      - Complex, usually overkill
      - Deep learning might be simpler!
"""

print(comparison)

## Why Ensembles Work 📊

### The Key Principle: Diversity + Combination

For an ensemble to work, you need:

1. **Diversity:** Models make different mistakes
   - Random Forests achieve this through randomness
   - Voting achieves this through different algorithms

2. **Reasonable Individual Performance:** Each model better than random guessing
   - If all models are terrible, averaging doesn't help!
   - Need weak learners that are still somewhat accurate

3. **Good Combination Strategy:**
   - Voting: Simple majority vote
   - Averaging: Good for regression
   - Weighted: Different models have different weights

### The Math (Simplified)

```
If 100 independent models each 55% accurate (barely better than random):
Ensemble accuracy ≈ 99%+!

Because errors don't correlate - when one gets it wrong,
others usually get it right. Averaging fixes this.
```

## Key Takeaways 🎯

✅ Ensemble methods combine multiple models for better predictions

✅ **Random Forest:** Parallel trees, reduces variance, great default choice

✅ **Gradient Boosting/XGBoost:** Sequential trees fixing mistakes, often best accuracy

✅ **Voting:** Combine different algorithm types for interpretability

✅ Ensembles outperform individual models due to diversity + aggregation

✅ Feature importance helps explain what models learned

## Practice Exercise 💪

1. **Compare models:** Build RF, XGBoost, and Voting on iris - which wins?
2. **Feature analysis:** Which iris features matter most? Why?
3. **Hyperparameter tuning:** Use GridSearchCV to tune Random Forest
4. **Ensemble depth:** Does ensemble perform better than any single tree?
5. **Regression:** Build ensemble for a regression task

Challenge: Build voting classifier mixing different algorithms, then beat it with XGBoost!

## Next Steps 🚀

→ **Feature Engineering:** How to create better features (huge impact!)

→ **Model Selection & Validation:** Choose best model, avoid overfitting

→ **Projects:** Apply to real datasets!

→ **Advanced Boosting:** LightGBM, CatBoost (even faster than XGBoost)